# Lab 07 – Task 2: ANN for handwritten digits (multi-class) – compare architectures
Requires `pip install tensorflow`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
tf.random.set_seed(42); np.random.seed(42)
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits

In [ ]:
d = load_digits()
Xd, yd = d.data / 16.0, d.target
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.2, random_state=42, stratify=yd)

def build(hidden, act):
    m = keras.Sequential([keras.layers.Input(shape=(64,))] + [keras.layers.Dense(h, activation=act) for h in hidden] + [keras.layers.Dense(10, activation='softmax')])
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

results = {}
for name, hidden, act in [('1x32 relu',[32],'relu'), ('1x128 relu',[128],'relu'), ('2x64 relu',[64,64],'relu'), ('2x64 tanh',[64,64],'tanh'), ('2x64 sigmoid',[64,64],'sigmoid')]:
    m = build(hidden, act)
    h = m.fit(Xtr, ytr, epochs=30, batch_size=32, validation_split=0.1, verbose=0)
    results[name] = (m.evaluate(Xte, yte, verbose=0)[1], h)
    print(f'{name:14s} test acc = {results[name][0]:.4f}')

In [ ]:
for n,(a,h) in results.items(): plt.plot(h.history['val_accuracy'], label=n)
plt.legend(); plt.xlabel('Epoch'); plt.ylabel('Validation accuracy'); plt.show()
final = build([64,64],'relu'); final.fit(Xtr, ytr, epochs=30, batch_size=32, verbose=0)
yp = final.predict(Xte).argmax(axis=1)
print(classification_report(yte, yp))
plt.figure(figsize=(8,6)); sns.heatmap(confusion_matrix(yte, yp), annot=True, fmt='d', cmap='Blues'); plt.xlabel('Predicted'); plt.ylabel('Truth'); plt.show()